# 04 - Search engine evaluation

This notebook is presentation-only. Index construction, TF-IDF/BM25 scoring, repeated queries, parameter sensitivity, and artifact export are performed by `scripts/run_search_experiment.py --config configs/sample.toml`. The report experiment uses the validated 50,000-review sample.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

def find_root():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise RuntimeError('Run the notebook inside the project repository')

ROOT = find_root()
RESULT_DIR = ROOT / 'artifacts' / 'search_engine' / 'sample'

manifest = json.loads((RESULT_DIR / 'run_manifest.json').read_text(encoding='utf-8'))
comparison = pd.read_csv(RESULT_DIR / 'comparison.csv')
rankings = pd.read_csv(RESULT_DIR / 'rankings.csv')
parameters = pd.read_csv(RESULT_DIR / 'bm25_parameters.csv')
assert manifest['status'] == 'COMPLETED'
assert manifest['ranked_queries'] >= 5
assert set(rankings['ranker']) == {'tfidf', 'bm25'}
print(f"Displaying: {RESULT_DIR.relative_to(ROOT)}")
print(f"Documents: {manifest['index_stats']['document_count']:,}")


## Index and experiment summary

In [ ]:
stats = manifest['index_stats']
pd.DataFrame({
    'metric': ['documents', 'vocabulary', 'indexed terms', 'average document length',
               'build seconds', 'experiment seconds'],
    'value': [stats['document_count'], stats['vocabulary_size'], stats['total_terms'],
              stats['avgdl'], manifest['build_seconds'], manifest['experiment_seconds']],
})


In [ ]:
comparison_display = comparison[[
    'query', 'overlap_count', 'jaccard', 'top1_match',
    'tfidf_all_query_terms_coverage', 'bm25_all_query_terms_coverage',
    'tfidf_median_ms', 'bm25_median_ms',
]].copy()
comparison_display.style.format({
    'jaccard': '{:.3f}',
    'tfidf_all_query_terms_coverage': '{:.2f}',
    'bm25_all_query_terms_coverage': '{:.2f}',
    'tfidf_median_ms': '{:.1f}',
    'bm25_median_ms': '{:.1f}',
})


In [ ]:
sns.set_theme(style='whitegrid')
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
sns.barplot(data=comparison, y='query', x='jaccard', color='#4C78A8', ax=axes[0])
axes[0].set_title('Top-10 overlap between rankers')
axes[0].set_xlabel('Jaccard similarity')
axes[0].set_ylabel('')
latency = comparison.melt(
    id_vars='query', value_vars=['tfidf_median_ms', 'bm25_median_ms'],
    var_name='ranker', value_name='median_ms')
latency['ranker'] = latency['ranker'].str.replace('_median_ms', '', regex=False).str.upper()
sns.barplot(data=latency, y='query', x='median_ms', hue='ranker', ax=axes[1])
axes[1].set_title('Warm query latency')
axes[1].set_xlabel('Median milliseconds')
axes[1].set_ylabel('')
fig.tight_layout()
plt.show()


## Side-by-side rankings

The table below exposes representative retrieved documents rather than treating overlap as relevance. Change `SELECTED_QUERY` to inspect any configured query.

In [ ]:
SELECTED_QUERY = comparison.iloc[0]['query']
top = rankings[(rankings['query'] == SELECTED_QUERY) & (rankings['rank'] <= 5)].copy()
top[['ranker', 'rank', 'score', 'business_name', 'stars',
     'document_length', 'matched_query_terms', 'text_excerpt']]


## BM25 parameter sensitivity

Jaccard similarity is measured against the default `k1=1.5, b=0.75` ranking. It describes ranking stability, not relevance quality.

In [ ]:
parameter_summary = (parameters.groupby(['k1', 'b'], as_index=False)
                     .agg(mean_jaccard=('default_jaccard', 'mean'),
                          top1_match_rate=('top1_matches_default', 'mean'),
                          mean_document_length=('mean_document_length', 'mean')))
parameter_summary.style.format({
    'mean_jaccard': '{:.3f}', 'top1_match_rate': '{:.2f}',
    'mean_document_length': '{:.1f}',
})


## Interpretation constraints

TF-IDF cosine normalizes complete document and query vectors, whereas BM25 saturates repeated term frequency and explicitly normalizes document length. Different Top-10 lists therefore demonstrate different ranking behaviour, not that either model is more relevant. No human relevance judgements were collected, so overlap, term coverage, latency, and parameter sensitivity must not be reported as precision, recall, or model accuracy.